# Experiment 4 - Part 2: The RAG Chain with LangChain and Gemini

**Course:** HAAI302COM - Artificial Intelligence Laboratory

In Part 1 we built a searchable knowledge base. Here we connect it to a language model:

```
question --> retriever (Chroma, top-k chunks) --> prompt (context + question) --> Gemini --> answer + sources
```

LangChain expresses this as a **chain**, built by joining small components with the `|` operator (LangChain Expression Language, LCEL). Each component is a `Runnable` with the same `.invoke()` method, so we can run and inspect every piece on its own before joining them.

We use Gemini through its OpenAI-compatible endpoint (`ChatOpenAI` with a Gemini `base_url`), as in Experiment 3.

## 0. Setup

`llm_setup.py` (in this folder) holds two small helpers: `get_llm()` builds the Gemini chat model, and `enable_cache()` stores every response in `llm_cache.json`. With the cache, re-running this notebook makes **no API calls** and gives identical answers. Delete `llm_cache.json` to call Gemini afresh.

In [1]:
import os, re, json, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

from llm_setup import get_llm, enable_cache
cache = enable_cache("llm_cache.json")
llm = get_llm("gemini-2.5-flash-lite", temperature=0.0)   # temperature 0: the most likely wording every time
print("cached responses on disk:", len(cache.data))

cached responses on disk: 25


Now reopen the vector store from Part 1. If `chroma_db/` is missing (for example on a fresh Colab), run notebook 01 first.

In [2]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store = Chroma(collection_name="rag_papers", embedding_function=embedder, persist_directory="chroma_db")
print("chunks in store:", store._collection.count())

qa = json.load(open("qa_set.json"))          # 12 answerable questions with reference answers, 2 unanswerable

chunks in store: 237


## 1. Baseline: ask the model without any documents

Before adding retrieval we need to know what the model does on its own. We ask four questions whose answers are written in the two papers.

In [3]:
picks = [0, 3, 4, 9]                                   # indices into qa["answerable"]
rows = []
for i in picks:
    item = qa["answerable"][i]
    rows.append({"question": item["question"], "no_rag_answer": llm.invoke(item["question"]).content})
base = pd.DataFrame(rows)
for _, r in base.iterrows():
    print("Q:", r.question, "\nA:", r.no_rag_answer[:420].replace("\n", " "), "\n")

Q: What are the two RAG formulations compared in the RAG paper, and how do they differ? 
A: The RAG paper (Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks) introduces and compares two primary formulations of RAG:  1.  **RAG-Sequence (RAG-S)** 2.  **RAG-Token (RAG-T)**  Here's a breakdown of how they differ:  ### 1. RAG-Sequence (RAG-S)  **Core Idea:** RAG-S treats the retrieved documents as **additional input to the generator**. It essentially concatenates the retrieved text with the origin 

Q: How was the Wikipedia knowledge source prepared for the RAG index? 
A: The preparation of the Wikipedia knowledge source for a Retrieval Augmented Generation (RAG) index involves several key steps, aiming to transform raw Wikipedia content into a format that can be efficiently searched and used by a RAG system. Here's a breakdown of the typical process:  **1. Data Acquisition and Extraction:**  *   **Downloading Wikipedia Dumps:** The most common method is to download the offic

**What to notice:** the answers are long, confident and generic. For the Wikipedia question the model describes how a *typical* RAG system might prepare Wikipedia, not what *this paper* did, and for the hot-swapping question it explains the idea in general terms but gives none of the paper's numbers (70% and 68% accuracy, 2016 and 2018 indexes). The model cannot cite a page, and nothing in the answer signals that it is speaking from memory. RAG addresses all three problems.

## 2. The retriever

A vector store becomes a **retriever** with `.as_retriever()`. A retriever is a Runnable: give it a string, get back a list of `Document` objects. The arguments are:

| Argument | Meaning |
|---|---|
| `search_type` | `"similarity"` (default), `"mmr"`, or `"similarity_score_threshold"` |
| `search_kwargs={"k": ...}` | How many chunks to return (also `fetch_k`, `lambda_mult`, `filter`, `score_threshold`) |

In [4]:
retriever = store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
question = qa["answerable"][4]["question"]
docs = retriever.invoke(question)

print(question)
print(type(retriever).__name__, "->", len(docs), "documents")

What does the index hot-swapping experiment show in the RAG paper?
VectorStoreRetriever -> 4 documents


In [5]:
pd.DataFrame({
    "rank": range(1, len(docs) + 1),
    "source": [d.metadata["source"].split("/")[-1] for d in docs],
    "page": [d.metadata["page"] + 1 for d in docs],        # page_label style: first page = 1
    "chars": [len(d.page_content) for d in docs],
    "text": [d.page_content[:80].replace("\n", " ") for d in docs],
})

,rank,source,page,chars,text
0,1,rag_lewis_2020.pdf,7,768,"RAG’s retriever with a fixed BM25 system, and use BM25 retrieval scores as logit"
1,2,rag_lewis_2020.pdf,17,480,and easier to use implementation. This version is also open-sourced. We also com
2,3,rag_lewis_2020.pdf,19,531,and the RAG model would perform equivalently to BART. The collapse could be due
3,4,rag_lewis_2020.pdf,17,748,their annotations were removed from the results. C Training setup Details We tra


**What to notice:** the retriever returned the four closest chunks with their metadata, best first. Read the text column: only rank 1 (page 7) is really about hot-swapping; ranks 2 to 4 come from the paper's appendix and are noise. The sentence we need is cut off at the end of page 7, and its numbers continue on the next page, because Part 1 split page by page. These two facts, noisy neighbours and passages cut at page boundaries, will show up in the RAGAs scores later.\n\nThis list is the *only* information the LLM may use, so if the right chunk is not here the answer will be wrong however good the LLM is. That is why RAGAs separates **retrieval metrics** (context precision, context recall) from **generation metrics** (faithfulness, answer relevancy).

## 3. The prompt

The prompt is where we tell the model to behave like a RAG system. A `ChatPromptTemplate` has named placeholders; `.invoke({...})` fills them and returns the messages that will be sent.

The system message does the important work: **answer only from the context, and admit when the context is not enough**. Without that sentence the model will happily fall back on its memory.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM = (
    "You answer questions about research papers using ONLY the context provided. "
    "If the context does not contain the answer, reply exactly: I don't know based on the provided documents. "
    "Answer in at most three sentences."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
print(prompt.input_variables)

['context', 'question']


The retriever returns `Document` objects but the prompt wants one string. A small formatting function joins the chunks and tags each with its file and page, so the model (and we) can see where each passage came from.

In [7]:
def format_docs(docs):
    # one block per chunk, labelled with a number, file name and page
    return "\n\n".join(
        f"[{i}] ({d.metadata['source'].split('/')[-1]}, p.{d.metadata['page'] + 1})\n{d.page_content}"
        for i, d in enumerate(docs, 1)
    )

context = format_docs(docs)
print(context[:600])

[1] (rag_lewis_2020.pdf, p.7)
RAG’s retriever with a fixed BM25 system, and use BM25 retrieval scores as logits when calculating
p(z|x). Table 6 shows the results. For FEVER, BM25 performs best, perhaps since FEVER claims are
heavily entity-centric and thus well-suited for word overlap-based retrieval. Differentiable retrieval
improves results on all other tasks, especially for Open-Domain QA, where it is crucial.
Index hot-swapping An advantage of non-parametric memory models like RAG is that knowledge
can be easily updated at test time. Parametric-only models like T5 or BART need further tra


In [8]:
messages = prompt.invoke({"context": context, "question": question}).to_messages()
for m in messages:
    print(f"--- {m.type} ({len(m.content)} characters)")
print(messages[0].content)

--- system (215 characters)
--- human (2743 characters)
You answer questions about research papers using ONLY the context provided. If the context does not contain the answer, reply exactly: I don't know based on the provided documents. Answer in at most three sentences.


**What to notice:** two messages will be sent: the fixed instruction (system) and the user message containing all four chunks plus the question. The user message is about 2,700 characters, which is why the number of retrieved chunks `k` directly drives the cost and latency of every answer. We measure that in section 6.

## 4. The chain

Now we join the pieces with `|`. Reading left to right: the dictionary builds the two prompt variables (the retriever output goes through `format_docs`; the question passes through unchanged), then the prompt, the model, and `StrOutputParser` which turns the model's message object into a plain string.

In [9]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print(chain.invoke(question))

The index hot-swapping experiment demonstrates that knowledge in RAG can be easily updated at test time, unlike parametric-only models like T5 or BART which require further training. This is shown by comparing RAG outputs using a December 2016 Wikipedia dump index to outputs from a newer index.


**What to notice:** compare this answer to the baseline answer for the same question in section 1. It is now short, tied to the paper (it mentions the December 2016 index and T5/BART) and it says nothing it cannot support. It is also incomplete: the reference answer contains the 70% and 68% figures, which were not in the retrieved text. The chain is a normal Runnable, so `chain.invoke`, `chain.batch` and `chain.stream` all work.

For evaluation (Part 3) and for user interfaces we also need the retrieved chunks next to the answer. `RunnablePassthrough.assign` adds new keys to a dictionary flowing through the chain without removing the old ones.

In [10]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

rag = (
    RunnablePassthrough.assign(docs=itemgetter("question") | retriever)              # step 1: retrieve
    .assign(answer=RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
                   | prompt | llm | StrOutputParser())                                # step 2: generate
)
out = rag.invoke({"question": question})
print("keys:", list(out))
print("answer:", out["answer"])

keys: ['question', 'docs', 'answer']
answer: The index hot-swapping experiment demonstrates that knowledge in RAG can be easily updated at test time, unlike parametric-only models like T5 or BART which require further training. This is shown by comparing RAG outputs using a December 2016 Wikipedia dump index to outputs from a newer index.


In [11]:
def cite(out):
    # the sources behind an answer, as a list of "file p.page"
    return sorted({f"{d.metadata['source'].split('/')[-1]} p.{d.metadata['page'] + 1}" for d in out["docs"]})

print(cite(out))

['rag_lewis_2020.pdf p.17', 'rag_lewis_2020.pdf p.19', 'rag_lewis_2020.pdf p.7']


**What to notice:** the output dictionary has the original `question`, the `docs` that were retrieved and the `answer`. Because we kept the documents, we can show the user *where* the answer came from, which is the main trust advantage of RAG over a bare LLM. We reuse this exact structure for RAGAs.

## 5. Question set: RAG against the baseline

We run 4 questions through the RAG chain and put the baseline answer, the RAG answer, the reference and the sources side by side.

In [12]:
rows = []
for i in picks:
    item = qa["answerable"][i]
    o = rag.invoke({"question": item["question"]})
    rows.append({"question": item["question"][:60], "rag_answer": o["answer"], "reference": item["reference"], "sources": cite(o)})
rag_df = pd.DataFrame(rows)
for _, r in rag_df.iterrows():
    print("Q:", r.question, "\nRAG:", r.rag_answer.replace("\n", " "), "\nREF:", r.reference, "\nSRC:", r.sources, "\n")

Q: What are the two RAG formulations compared in the RAG paper, 
RAG: The two RAG formulations compared are RAG-Token and RAG-Sequence. RAG-Sequence is an improved version that is easier to use and has been open-sourced. 
REF: RAG-Sequence uses the same retrieved document to generate the whole output sequence, while RAG-Token can use a different retrieved document for each target token. 
SRC: ['rag_lewis_2020.pdf p.10', 'rag_lewis_2020.pdf p.17', 'rag_lewis_2020.pdf p.5', 'rag_lewis_2020.pdf p.8'] 

Q: How was the Wikipedia knowledge source prepared for the RAG  
RAG: I don't know based on the provided documents. 
REF: The December 2018 Wikipedia dump was split into disjoint 100-word chunks, giving 21M documents, which were embedded and stored in a FAISS MIPS index with an HNSW approximation. 
SRC: ['rag_lewis_2020.pdf p.10', 'rag_lewis_2020.pdf p.5', 'rag_lewis_2020.pdf p.7'] 

Q: What does the index hot-swapping experiment show in the RAG  
RAG: The index hot-swapping experiment demo

**What to notice:** four questions show four different outcomes, and each one points to a different part of the pipeline.

- *Two formulations:* the answer names RAG-Token and RAG-Sequence but then adds a claim about RAG-Sequence being "an improved version that is easier to use". That sentence comes from an appendix chunk about a different topic, and it does not answer how the two differ. The model was faithful to **a** chunk, but the wrong one was retrieved, so the answer is wrong. This is a retrieval problem.
- *Wikipedia preparation:* "I don't know". The guardrail worked, but the answer exists in the paper. The right chunk was not retrieved. This is also a retrieval problem, and it is the honest failure mode.
- *Hot-swapping:* correct but without the numbers (cut at the page boundary, see section 2).
- *LoRA latency:* correct and complete. The retrieved chunks contained the key sentence.

Remember: a wrong answer in RAG is more often a **retrieval** failure than a language-model failure. RAGAs will let us tell the two apart with numbers.

## 6. Experiment: how many chunks (`k`) should we send?

More chunks means a better chance the answer is included but also more tokens (cost, latency) and more irrelevant text that can distract the model. We vary `k` for one question and record the prompt size reported by the API in `usage_metadata`.

In [13]:
import time
q = qa["answerable"][2]["question"]      # generator model and parameter count
rows = []
for k in [1, 2, 4, 8]:
    r = store.as_retriever(search_kwargs={"k": k})
    d = r.invoke(q)
    t0 = time.time()
    msg = (prompt | llm).invoke({"context": format_docs(d), "question": q})     # AIMessage, so we can read usage
    rows.append({"k": k, "prompt_tokens": msg.usage_metadata["input_tokens"],
                 "has_BART_large": any("bart-large" in x.page_content.lower() for x in d),
                 "answer": msg.content.replace("\n", " ")[:150]})
pd.DataFrame(rows)

,k,prompt_tokens,has_BART_large,answer
0,1,215,False,I don't know based on the provided documents.
1,2,463,False,The RAG-Sequence model uses the retrieved document as a single latent variable. The RAG-Token mo...
2,4,722,True,"The RAG models use BART-large as the generator, which has 406M parameters. The RAG models also c..."
3,8,1793,True,The RAG models use a BART-large generator with 406M parameters. The RAG models also contain trai...


**What to notice:** the column `has_BART_large` shows whether the retrieved text actually contained the answer.

- `k=1` (about 215 prompt tokens): the one chunk does not contain it, so the model correctly says "I don't know".
- `k=2`: still no BART-large chunk. The model describes RAG-Sequence and RAG-Token instead and admits the parameter count is not given. Honest, but it is answering a different question.
- `k=4` (about 720 tokens): the right chunk arrives and the answer is correct. The model says 406M, the appendix figure for BART-large's trainable parameters (the body text rounds it to 400M, our reference).
- `k=8` (about 1,800 tokens): the same correct answer for 2.5 times the tokens.

Answer quality jumps once the relevant chunk enters the prompt, and extra chunks after that only add cost and latency. The best `k` is the smallest one that reliably contains the evidence. We choose it with measurements in Notebook 03.

## 7. Unanswerable questions: the guardrail matters

A good RAG system must say "I don't know" when the documents do not contain the answer. We ask two questions that are *not* in either paper, once with our strict system prompt and once with a naive prompt that only says "use the context".

In [14]:
naive = ChatPromptTemplate.from_messages([("human", "Use the context to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}")])
strict_chain = prompt | llm | StrOutputParser()
naive_chain = naive | llm | StrOutputParser()

rows = []
for q in qa["unanswerable"]:
    ctx = format_docs(retriever.invoke(q))
    rows.append({"question": q, "strict_prompt": strict_chain.invoke({"context": ctx, "question": q}).replace("\n", " ")[:200],
                 "naive_prompt": naive_chain.invoke({"context": ctx, "question": q}).replace("\n", " ")[:200]})
pd.DataFrame(rows)

,question,strict_prompt,naive_prompt
0,What is the price per million tokens of the GPT-4 API?,I don't know based on the provided documents.,"I am sorry, but the provided context does not contain information about the price per million to..."
1,What accuracy does the RAG model achieve on the HotpotQA dataset?,I don't know based on the provided documents.,"Based on the provided context, the accuracy of the RAG model on the HotpotQA dataset is not expl..."
2,How well does LoRA perform when applied to the LLaMA-2 70B model?,I don't know based on the provided documents.,"Based on the provided context, there is no information about how LoRA performs when applied to t..."


**What to notice:** with the strict prompt all three questions get the same fixed sentence, which is easy for code to detect (for example to route the user to a human or to a web search). The naive prompt also avoids a wrong answer on the first and third question, but on the HotpotQA question it starts guessing from the table it can see: it says "WQ likely refers to WikiHop" and "TQA (Textual Question Answering)", neither of which is correct (they are WebQuestions and TriviaQA). It never claims a made-up number, but speculation like this is how hallucinations start. A strong model like Gemini needs the strict prompt less than a small local model does, yet the sentence costs nothing. Keep it.

## 8. Streaming and batching

Every chain supports `.stream()` (tokens as they are produced, which makes a chat UI feel fast) and `.batch()` (several inputs in parallel threads).

In [15]:
for token in chain.stream("What is MIPS used for in RAG?"):
    print(token, end="", flush=True)

I

 don't know based on the provided documents.

In [16]:
questions = [qa["answerable"][i]["question"] for i in (1, 2, 5)]
answers = chain.batch(questions, config={"max_concurrency": 3})
for q, a in zip(questions, answers):
    print("Q:", q[:70], "\nA:", a.replace("\n", " ")[:200], "\n")

Q: Which retriever does the RAG model use, and what architecture is it? 
A: The RAG model uses a retriever pη(z|x) with parameters η that returns distributions over text passages given a query x. The provided documents do not specify the architecture of this retriever. 

Q: Which generator model does RAG use and how many parameters does it hav 
A: The RAG models use BART-large as the generator, which has 406M parameters. The RAG models also contain trainable parameters for the BERT-base query and document encoder of DPR, with 110M parameters ea 

Q: What search problem is solved to find the top-K documents in RAG? 
A: Calculating the top-k documents involves a Maximum Inner Product Search (MIPS) problem. This problem can be approximately solved in sub-linear time. The RAG model uses a pre-trained bi-encoder from DP 



**What to notice:** streaming prints the answer progressively; batching returns the answers in the same order as the questions. Both come for free because the chain is built from Runnables.

## Summary

| Component | LangChain object | What it does | Where to look when it fails |
|---|---|---|---|
| Retriever | `store.as_retriever(search_kwargs={"k": ...})` | Finds the chunks | `retriever.invoke(q)`: is the right chunk in the list? |
| Prompt | `ChatPromptTemplate` | Instructs the model and inserts the context | `prompt.invoke(...)`: read the final messages |
| Model | `ChatOpenAI` (Gemini endpoint) | Writes the answer | `usage_metadata`: tokens and cost |
| Parser | `StrOutputParser` | Message to string | - |
| Chain | `a \| b \| c` | Connects everything | `chain.invoke / stream / batch` |

**Next:** Notebook 03 measures all of this with RAGAs: faithfulness, answer relevancy, context precision and context recall.